In [5]:
import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl

def calculate_custom_apartment_price():
    #Ініціалізація змінних
    time = ctrl.Antecedent(np.arange(0, 101, 1), 'time')
    area = ctrl.Antecedent(np.arange(20, 151, 1), 'area')
    quality = ctrl.Antecedent(np.arange(1, 11, 1), 'quality')
    price = ctrl.Consequent(np.arange(10, 201, 1), 'price')

    #Функції належності
    time['near'] = fuzzy.trimf(time.universe, [0, 0, 30])
    time['medium'] = fuzzy.trimf(time.universe, [15, 45, 75])
    time['far'] = fuzzy.trimf(time.universe, [60, 100, 100])

    area['small'] = fuzzy.trimf(area.universe, [20, 20, 60])
    area['medium'] = fuzzy.trimf(area.universe, [40, 80, 120])
    area['large'] = fuzzy.trimf(area.universe, [90, 150, 150])

    quality.automf(3, names=['low', 'average', 'high'])

    price['cheap'] = fuzzy.trimf(price.universe, [10, 10, 60])
    price['affordable'] = fuzzy.trimf(price.universe, [40, 90, 140])
    price['expensive'] = fuzzy.trimf(price.universe, [110, 200, 200])

    #База правил по мамдані
    rules = [
        ctrl.Rule(time['near'] & area['large'] & quality['high'], price['expensive']),
        ctrl.Rule(time['near'] | time['medium'] & area['medium'] & quality['average'], price['affordable']),
        ctrl.Rule(time['far'] | area['small'] | quality['low'], price['cheap']),
        ctrl.Rule(area['large'] & quality['low'], price['affordable']),
        ctrl.Rule(time['near'] & area['small'] & quality['high'], price['affordable'])
    ]

    # симуляція
    pricing_ctrl = ctrl.ControlSystem(rules)
    simulation = ctrl.ControlSystemSimulation(pricing_ctrl)

    print("=== Інтерактивна оцінка вартості квартири (Fuzzy Logic) ===\n")

    # цикл
    while True:
        try:
            print("-" * 40)
            t_val = float(input("Введіть час до центру (хв, 0-100): "))
            a_val = float(input("Введіть площу квартири (кв.м, 20-150): "))
            q_val = float(input("Введіть якість житла (1-10): "))


            simulation.input['time'] = t_val
            simulation.input['area'] = a_val
            simulation.input['quality'] = q_val

            # 6. Дефазифікація
            simulation.compute()
            estimated_price = simulation.output['price']

            print(f"\n-> Орієнтовна вартість: {estimated_price:.1f} тис. $\n")

            cont = input("Оцінити ще одну квартиру? (т/н): ").strip().lower()
            if cont not in ['т', 'так', 'y', 'yes']:
                print("Роботу завершено.")
                break

        except ValueError:
            print("Помилка: будь ласка, вводьте лише числа.")

if __name__ == "__main__":
    calculate_custom_apartment_price()

=== Інтерактивна оцінка вартості квартири (Fuzzy Logic) ===

----------------------------------------
Введіть час до центру (хв, 0-100): 10
Введіть площу квартири (кв.м, 20-150): 78
Введіть якість житла (1-10): 7

-> Орієнтовна вартість: 90.0 тис. $



KeyboardInterrupt: Interrupted by user